&nbsp;

## Antisense Transcript (UP-DOWN) Survey

- 小野さんが調べていたような、UP-DOWN or DOWN-UP の事例を調査する

In [1]:
import datetime
import os
import pathlib

import polars as pl
import polars.selectors as cs
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

## Prepare HN-score + fanta.bio

In [2]:
HNscore_all_fantabio = pl.read_parquet(
    "../../out/fantabio/parquet_file/HNscore_transcript_all_fantabio_v261001.parquet"
).filter(
    pl.col("cre_id").is_not_null() # fanta.bio の CRE が紐づかなかった場合は、今回は考慮しない
)

display(HNscore_all_fantabio)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28106""","""chr1""",161524161,"""161524546""","""cp4@HSPA6""","[""ENST00000309758.6"", ""NM_002155.5"", … ""AK295212""]","""0""",0,"[{""EH38E2843855"",""PLS""}]","[{""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX3322007"",560}]}, {""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX13551280"",511}]}, … {""MED12"",511,[{""SRX10852993"",511}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr1:161494328..161494332,+;hg_6264.1"",""p1@HSPA6""}]",null,0.943234,"""PLA""","""204,0,0""","""red"""
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_456076""","""chr1""",161524872,"""161524930""","""FCHS_456076""","[""KJ897015""]","""-214""",-332,"[{""EH38E2843856"",""pELS""}, {""EH38E3981534"",""PLS""}]","[{""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX12008350"",1465}]}, {""ZNF687"",2211,[{""SRX10478566"",2211}]}, … {""NR5A2"",516,[{""SRX10478059"",516}]}]","[{""HS_00013404"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00013405"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,1.0,"""ELA""","""255,202,74""","""yellow"""
"""ENST00000679223.2""",338,8,60,330,222,1,28,221,"""DNAJB1""","""HGNC:5270""","""protein_coding""","""ENSG00000132002.10""","[""alternative_5_UTR"", ""inferred_transcript_model"", … ""CCDS""]","""-""",5,null,null,"[""polya_site""]","""ensembl_havana_gene_homo_sapiens""","[""alternative 5' UTR"", ""infer

&nbsp;

&nbsp;

## Prepare: ENCODE-rE2G Result (Heat Stress)

In [3]:
encode_rE2G_hs_result = pl.read_parquet(
    "../../out/ENCODE_rE2G/parquet_file/encode_re2g_hs_join_v261003.parquet"
).filter(
    pl.col("isSelfPromoter") == False
)


encode_rE2G_hs_result_unique_count = encode_rE2G_hs_result.group_by(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class"
).agg(
    pl.col("re2g_dataset").unique().alias("re2g_dataset"),
    pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").alias("enhancer_cre_overlap_frac_of_fantabio_cre"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").min().alias("enhancer_cre_overlap_frac_of_fantabio_cre_min"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").max().alias("enhancer_cre_overlap_frac_of_fantabio_cre_max"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").mean().alias("enhancer_cre_overlap_frac_of_fantabio_cre_mean"), # 該当するデータセットで同定された enhancer と fantabio の CRE の重複率の平均値

    pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
    pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
    pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"), # 該当するデータセットで同定された enhancer と target promoter の相互作用のスコアの平均値

    # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
    # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
    # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
    # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
).sort(
    by=["re2g_dataset_count"],
    descending=True
)

print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): {encode_rE2G_hs_result_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
display(encode_rE2G_hs_result_unique_count)

ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): 368398


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64
"""FCHS_235818""","""FCHS_235818""","""PLA""","""FCHS_235801""","""cp3@NR4A2""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.744928, 0.597101, … 1.0]",0.507246,1.0,0.880952,"[0.861185, 0.881105, … 0.899437]",0.551271,0.981323,0.840312
"""FCHS_512792""","""FCHS_512792""","""ELA""","""FCHS_200631""","""cp3@ECSIT""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 1.0, … 1.0]",1.0,1.0,1.0,"[0.41146, 0.594062, … 0.718398]",0.328073,0.776167,0.551761
"""FCHS_359520""","""cp5@GCLC""","""PLA""","""FCHS_359518""","""cp3@GCLC""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.853211, 0.851376, … 0.921101]",0.831193,1.0,0.891088,"[0.93902, 0.942644, … 0.950412]",0.925775,0.971776,0.955644
"""FCHS_214284""","""cp7@MZF1-AS1""","""PLA""","""FCHS_214277""","""cp4@MZF1,cp8@MZF1-AS1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 0.841509, … 0.739623]",0.739623,1.0,0.913073,"[0.851593, 0.799192, … 0.700261]",0.638386,0.890422,0.821078
"""FCHS_159419""","""cp5@ZNF19""","""PLA""","""FCHS_159415""","""cp3@ZNF23""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 1.0, … 1.0]",1.0,1.0,1.0,"[0.877818, 0.728854, … 0.313979]",0.313979,0.899552,0.759179
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_219323""","""FCHS_219323""","""ELA""","""FCHS_219301""","""cp3@SRD5A2""","""PLA""","[""GSE154744_K562_42C_12_5min""]",1,[1.0],1.0,1.0,1.0,[0.308804],0.308804,0.308804,0.308804
"""FCHS_477267""","""FCHS_477267""","""ELA""","""FCHS_94051""","""cp2@LIMA1""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[1.0],1.0,1.0,1.0,[0.390645],0.390645,0.390645,0.390645
"""FCHS_386554""","""FCHS_386554""","""ELA""","""FCHS_386532""","""cp1@MEPCE,cp2@ZCWPW1""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,"[1.0, 1.0]",1.0,1.0,1.0,"[0.224512, 0.224071]",0.224071,0.224512,0.224292


&nbsp;

&nbsp;

&nbsp;

## Preparation: antisense transcript pair

- [../06_gene_research/06_6_antisense_RNA.ipynb](../06_gene_research/06_6_antisense_RNA.ipynb) で作成したファイルを使用する

In [4]:
bedfile_intersect_antisense_join = pl.read_parquet(
    "../../out/transcript_classification/antisense_transcript/antisense_transcript_join_HNscore_fantabio_v261010.parquet"
)

display(bedfile_intersect_antisense_join)

chr_a,start_a,end_a,transcript_id_a,strand_a,chr_b,start_b,end_b,transcript_id_b,strand_b,num_match,frac_match_a,frac_match_b,gene_id_a,gene_symbol_a,transcript_type_a,HN2_a,HN2_stranded_a,cre_info_a,n_cre_a,gene_id_b,gene_symbol_b,transcript_type_b,HN2_b,HN2_stranded_b,cre_info_b,n_cre_b,n_shared_cre,n_shared_cre_ELA,n_shared_cre_PLA
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str,str,str,i64,i64,list[struct[2]],u32,str,str,str,i64,i64,list[struct[2]],u32,u32,u32,u32
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778746,810063,"""ENST00000412115.2""","""+""",31317,0.501899,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",9,12,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778757,810062,"""ENST00000670700.1""","""+""",31305,0.501707,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",0,3,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778759,810065,"""ENST00000657896.2""","""+""",31306,0.501723,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",10,4,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778771,810065,"""ENST00000669749.2""","""+""",31294,0.501531,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",2,2,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778795,810062,"""ENST00000658648.1""","""+""",31267,0.501098,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",7,2,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",155071502,155123074,"""ENST00000369462.5""","""+""","""chrX""",155065320,155147775,"""ENST00000362018.2""","""-""",51572,1.0,0.625456,"""ENSG00000185515.16""","""BRCC3""","""protein_coding""",-65,-41,"[{""FCHS_446828"",""PLA""}, {""FCHS_593833"",""PLA""}, {""FCHS_446830"",""ELA""}]",3,"""ENSG00000214827.12""","""MTCP1""","""protein_coding""",4,3,"[{""FCHS_446836"",""PLA""}]",1,0,0,0
"""chrX""",155071507,155123077,"""ENST00000330045.12""","""+""","""chrX""",155065162,155147937,"""ENST00000476116.1""","""-""",51570,1.0,0.623014,"""ENSG00000185515.16""","""BRCC3""","""protein_coding""",7,2,"[{""FCHS_446828"",""PLA""}, {""FCHS_593833"",""PLA""}, {""FCHS_446830"",""ELA""}]",3,"""ENSG00000214827.12""","""MTCP1""","""protein_coding_CDS_not_defined""",-9,4,"[{""FCHS_446836"",""PLA""}]",1,0,0,0
"""chrX""",155071507,155123077,"""ENST00000330045.12""","""+""","""chrX""",155065320,155147775,"""ENST00000362018.2""","""-""",51570,1.0,0.625432,"""ENSG00000185515.16""","""BRCC3""","""protein_coding""",7,2,"[{""FCHS_446828"",""PLA""}, {""FCHS_593833"",""PLA""}, {""FCHS_446830"",""ELA""}]",3,"""ENSG00000214827.12""","""MTCP1""","""protein_coding""",4,3,"[{""FCHS_446836"",""PLA""}]",1,0,0,0


In [5]:
# Canonical order so A-B and B-A collapse to one label.
pair_labels = (
    bedfile_intersect_antisense_join.select(
        pl.min_horizontal("transcript_id_a", "transcript_id_b").alias("id_lo"),
        pl.max_horizontal("transcript_id_a", "transcript_id_b").alias("id_hi"),
    )
    .unique(maintain_order=True)
    .select(pl.format("{} - {}", pl.col("id_lo"), pl.col("id_hi")).alias("pair"))
)

print(pair_labels.height)
pl.Config.set_tbl_rows(2)
display(pair_labels)

49887


pair
str
"""ENST00000412115.2 - ENST00000743835.1"""
…
"""ENST00000496301.6 - ENST00000728197.1"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. UP-DOWN or DOWN-UP (stringent)

- どちらかの transcript が HN-score (stranded) >= 50 かつ HN-score (stranded) <= -50 である場合をフィルタリング
- 注目しやすい transcript pair を探すための簡単な条件

In [11]:
antisense_up_down = bedfile_intersect_antisense_join.filter(
    ((pl.col("HN2_stranded_a") >= 50) & (pl.col("HN2_stranded_b") <= -50))
    | ((pl.col("HN2_stranded_a") <= -50) & (pl.col("HN2_stranded_b") >= 50))
)

pl.Config.set_tbl_rows(2)
display(antisense_up_down)

chr_a,start_a,end_a,transcript_id_a,strand_a,chr_b,start_b,end_b,transcript_id_b,strand_b,num_match,frac_match_a,frac_match_b,gene_id_a,gene_symbol_a,transcript_type_a,HN2_a,HN2_stranded_a,cre_info_a,n_cre_a,gene_id_b,gene_symbol_b,transcript_type_b,HN2_b,HN2_stranded_b,cre_info_b,n_cre_b,n_shared_cre,n_shared_cre_ELA,n_shared_cre_PLA
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str,str,str,i64,i64,list[struct[2]],u32,str,str,str,i64,i64,list[struct[2]],u32,u32,u32,u32
"""chr1""",19210329,19240709,"""ENST00000437898.4""","""+""","""chr1""",19218092,19251537,"""ENST00000911103.1""","""-""",22617,0.74447,0.676245,"""ENSG00000230424.4""","""EMC1-AS1""","""lncRNA""",159,110,"[{""FCHS_449315"",""ELA""}, {""FCHS_5474"",""PLA""}, {""FCHS_5475"",""ELA""}]",3,"""ENSG00000127463.17""","""EMC1""","""protein_coding""",-56,-51,"[{""FCHS_5487"",""PLA""}]",1,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr9""",136363956,136370893,"""ENST00000485975.1""","""-""","""chr9""",136363315,136368893,"""ENST00000772720.1""","""+""",4937,0.711691,0.885084,"""ENSG00000187796.17""","""CARD9""","""retained_intron""",145,128,"[{""FCHS_432852"",""ELA""}, {""FCHS_432853"",""ELA""}, {""FCHS_589395"",""ELA""}]",3,"""ENSG00000300558.1""",null,"""lncRNA""",-81,-54,"[{""FCHS_432846"",""PLA""}, {""FCHS_432847"",""PLA""}]",2,0,0,0


In [12]:
# Canonical order so A-B and B-A collapse to one label.
id_lo = pl.min_horizontal("transcript_id_a", "transcript_id_b")
a_is_lo = pl.col("transcript_id_a") == id_lo

pair_labels = (
    antisense_up_down.select(
        id_lo.alias("id_lo"),
        pl.max_horizontal("transcript_id_a", "transcript_id_b").alias("id_hi"),
        pl.when(a_is_lo).then(pl.col("gene_symbol_a")).otherwise(pl.col("gene_symbol_b")).alias("symbol_lo"),
        pl.when(a_is_lo).then(pl.col("gene_symbol_b")).otherwise(pl.col("gene_symbol_a")).alias("symbol_hi"),
    )
    .unique(subset=["id_lo", "id_hi"], maintain_order=True)
    .select(
        pl.format(
            "{} ({}) - {} ({})",
            pl.col("id_lo"),
            pl.col("symbol_lo").fill_null(""),
            pl.col("id_hi"),
            pl.col("symbol_hi").fill_null(""),
        ).alias("pair")
    )
)

print(pair_labels.height)
display(pair_labels)

37


pair
str
"""ENST00000437898.4 (EMC1-AS1) - ENST00000911103.1 (EMC1)"""
…
"""ENST00000485975.1 (CARD9) - ENST00000772720.1 ()"""


&nbsp;

&nbsp;

&nbsp;

## lncRNA - protein coding only

In [13]:
antisense_up_down_filter = antisense_up_down.filter(
    ((pl.col("transcript_type_a") == "lncRNA") | (pl.col("transcript_type_b") == "lncRNA"))
).filter(
    ((pl.col("transcript_type_a") == "protein_coding") | (pl.col("transcript_type_b") == "protein_coding"))
)

display(antisense_up_down_filter)

chr_a,start_a,end_a,transcript_id_a,strand_a,chr_b,start_b,end_b,transcript_id_b,strand_b,num_match,frac_match_a,frac_match_b,gene_id_a,gene_symbol_a,transcript_type_a,HN2_a,HN2_stranded_a,cre_info_a,n_cre_a,gene_id_b,gene_symbol_b,transcript_type_b,HN2_b,HN2_stranded_b,cre_info_b,n_cre_b,n_shared_cre,n_shared_cre_ELA,n_shared_cre_PLA
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str,str,str,i64,i64,list[struct[2]],u32,str,str,str,i64,i64,list[struct[2]],u32,u32,u32,u32
"""chr1""",19210329,19240709,"""ENST00000437898.4""","""+""","""chr1""",19218092,19251537,"""ENST00000911103.1""","""-""",22617,0.74447,0.676245,"""ENSG00000230424.4""","""EMC1-AS1""","""lncRNA""",159,110,"[{""FCHS_449315"",""ELA""}, {""FCHS_5474"",""PLA""}, {""FCHS_5475"",""ELA""}]",3,"""ENSG00000127463.17""","""EMC1""","""protein_coding""",-56,-51,"[{""FCHS_5487"",""PLA""}]",1,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr9""",136363955,136372928,"""ENST00000892163.1""","""-""","""chr9""",136363315,136368893,"""ENST00000772720.1""","""+""",4938,0.550318,0.885264,"""ENSG00000187796.17""","""CARD9""","""protein_coding""",106,101,"[{""FCHS_432854"",""ELA""}, {""FCHS_432855"",""ELA""}]",2,"""ENSG00000300558.1""",null,"""lncRNA""",-81,-54,"[{""FCHS_432846"",""PLA""}, {""FCHS_432847"",""PLA""}]",2,0,0,0
